# Article Analysis

Este notebook lê apenas os artefatos salvos pelos scripts de treino e gera tabelas e gráficos para o trabalho.

In [ ]:
from pathlib import Path
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt

RUN_ROOT = Path('/content/drive/MyDrive/cla_lora_runs')
RUN_NAMES = {
    'full_finetune': None,
    'lora': None,
    'adalora': None,
}
K = 4


In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def latest_run_dir(method):
    method_dir = RUN_ROOT / method
    run_dirs = [path for path in method_dir.iterdir() if path.is_dir()]
    return sorted(run_dirs)[-1]

def get_run_dir(method):
    if RUN_NAMES[method] is None:
        return latest_run_dir(method)
    return RUN_ROOT / method / RUN_NAMES[method]

def load_run(method):
    run_dir = get_run_dir(method)
    return {
        'run_dir': run_dir,
        'config': load_json(run_dir / 'config.json'),
        'summary': load_json(run_dir / 'summary.json'),
        'history': pd.read_csv(run_dir / 'train_history.csv'),
        'deltas': torch.load(run_dir / 'target_deltas.pt', map_location='cpu'),
        'svdvals': torch.load(run_dir / 'target_svdvals.pt', map_location='cpu'),
        'layer_stats': load_json(run_dir / 'layer_stats.json'),
    }

def principal_subspace(delta, side, k):
    u, s, vh = torch.linalg.svd(delta.float(), full_matrices=False)
    if s.numel() == 0:
        return None
    rank = min(k, s.numel())
    if side == 'left':
        return u[:, :rank]
    return vh[:rank, :].T

def subspace_similarity(delta_ref, delta_cmp, k):
    metrics = {}
    for side in ['left', 'right']:
        basis_ref = principal_subspace(delta_ref, side, k)
        basis_cmp = principal_subspace(delta_cmp, side, k)
        gram = basis_ref.T @ basis_cmp
        cosines = torch.linalg.svdvals(gram).clamp(0.0, 1.0)
        angles = torch.rad2deg(torch.acos(cosines))
        overlap = torch.linalg.matrix_norm(gram, ord='fro').item() ** 2 / gram.shape[0]
        metrics[f'{side}_mean_cosine'] = cosines.mean().item()
        metrics[f'{side}_max_angle_deg'] = angles.max().item()
        metrics[f'{side}_overlap'] = overlap
    return metrics


In [ ]:
runs = {
    'full_finetune': load_run('full_finetune'),
    'lora': load_run('lora'),
    'adalora': load_run('adalora'),
}
runs['full_finetune']['run_dir']

In [ ]:
summary_rows = []
for method, run in runs.items():
    summary = run['summary']
    summary_rows.append({
        'method': method,
        'trainable_params': summary['trainable_params'],
        'trainable_pct': summary['trainable_pct'],
        'final_train_loss': summary['final_train_loss'],
        'final_test_loss': summary['final_test_loss'],
        'total_time_sec': summary['total_time_sec'],
    })
pd.DataFrame(summary_rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for method, run in runs.items():
    history = run['history']
    axes[0].plot(history['epoch'], history['train_loss'], marker='o', label=method)
    axes[1].plot(history['epoch'], history['test_loss'], marker='o', label=method)
axes[0].set_title('Train loss por época')
axes[1].set_title('Test loss por época')
axes[0].legend()
axes[1].legend()
plt.tight_layout()

In [ ]:
attention_layers = sorted([
    name for name in runs['full_finetune']['deltas'].keys()
    if '.attn.c_attn' in name
])
attention_layers[:3]

In [ ]:
layer_name = attention_layers[0]
plt.figure(figsize=(8, 4))
for method, run in runs.items():
    svdvals = run['svdvals'][layer_name]
    plt.plot(range(1, len(svdvals) + 1), svdvals.numpy(), marker='o', label=method)
plt.yscale('log')
plt.title(f'Espectro singular: {layer_name}')
plt.xlabel('Índice')
plt.ylabel('Valor singular')
plt.legend()
plt.tight_layout()

In [ ]:
svd_rows = []
for method, run in runs.items():
    for layer_name, stats in run['layer_stats'].items():
        if '.attn.c_attn' in layer_name:
            svd_rows.append({
                'method': method,
                'layer': layer_name,
                'stable_rank': stats['stable_rank'],
                'energy_90_rank': stats['energy_90_rank'],
                'energy_95_rank': stats['energy_95_rank'],
            })
pd.DataFrame(svd_rows).head()

In [ ]:
similarity_rows = []
for method in ['lora', 'adalora']:
    for layer_name in attention_layers:
        metrics = subspace_similarity(
            runs['full_finetune']['deltas'][layer_name],
            runs[method]['deltas'][layer_name],
            K,
        )
        similarity_rows.append({
            'method': method,
            'layer': layer_name,
            **metrics,
        })
pd.DataFrame(similarity_rows).head()

In [ ]:
adalora_mlp_rows = []
for layer_name, stats in runs['adalora']['layer_stats'].items():
    if '.mlp.' in layer_name:
        adalora_mlp_rows.append({'layer': layer_name, **stats})
pd.DataFrame(adalora_mlp_rows).head()